In [1]:
# ---- MBE + GAPT ---- 
# 1. Modified GPT (with MBE calculation)
from src.gapt import GPTConfig, GPT
import torch 

config = GPTConfig(
    n_layer=4,
    n_head=4,
    n_embd=128,
)
    
model = GPT(config)
# model = model.to("cuda")
# model = torch.compile(model)

In [2]:
# heuristic rollout
import os, glob, itertools
from pathlib import Path

# MPS specific data loader functional (single device ver.)
# --------------------------------------------
def _load_data_shard(file: Path):
    header = torch.from_file(str(file), False, 256, dtype=torch.int32) # header is 256 int32
    assert header[0] == 20240520, "magic number mismatch in the data .bin file"
    assert header[1] == 1, "unsupported version"
    num_tokens = int(header[2]) # number of tokens (claimed)
    with file.open("rb", buffering=0) as f:
        tokens = torch.empty(num_tokens, dtype=torch.uint16, pin_memory=False) # MPS requires pin_memory=False
        f.seek(256 * 4)
        nbytes = f.readinto(tokens.numpy()) # avoid bytes->array copy by @YouJiacheng
        assert nbytes == 2 * num_tokens, "number of tokens read does not match header"
    return tokens

def data_generator(filename_pattern: str, sequence_length: int, device: str): 

    filename_pattern = "data/fineweb10B/fineweb_train_*.bin"
    files = [Path(file) for file in sorted(glob.glob(filename_pattern))]
    file_iter = itertools.cycle(files)
    tokens, pos = _load_data_shard(next(file_iter)), 0
    while True: 
        # Concern 1. Doesn't this means end-of-file is never reached?
        if pos + sequence_length + 1 >= len(tokens): # not enough data left -> load a new file
            tokens, pos = _load_data_shard(next(file_iter)), 0
        
        buf = tokens[pos : pos + sequence_length + 1]
        inputs = buf[None, :-1].to(device=device, dtype=torch.int32, non_blocking=True) # no sync on host side;
        targets = buf[None, 1:].to(device=device, dtype=torch.int64, non_blocking=True) 
        pos += sequence_length
        yield inputs, targets

data = _load_data_shard(Path("data/fineweb10B/fineweb_train_000002.bin"))
train_loader = data_generator(filename_pattern="data/fineweb10B/fineweb_train_*.bin", sequence_length=16, device="cpu")

In [5]:
import time
from src.mbe import patch_mbe
from src.gradtracker import GradientTracker

g = GradientTracker(model)

# --- forward propagation ---
input, target = next(train_loader)

model.enable_timing = True
output = model(input, target, attn_blocksize=256, patch_size=8)
model.enable_timing = False

# --- backward propagation --- 
# Question 1. if I backward pass on added MBE loss, will it has the same speed & memory usage? 
g.backward_with_tracking({"mbe_0": output["mbe_0"]}, retain_graph=True)
g.backward_with_tracking({"mbe_1": output["mbe_1"]}, retain_graph=True)
g.backward_with_tracking({"mbe_2": output["mbe_2"]}, retain_graph=True)
g.backward_with_tracking({"mbe": output["mbe_0"] + output["mbe_1"] + output["mbe_2"]}, retain_graph=True)
g.backward_with_tracking({"ce": output["entropy"]})


⏱️  Forward Pass Timing Breakdown (ms)
Setup (mask + embed):         3.56 ms  ( 11.6%)
Encoder Forward:             14.01 ms  ( 45.7%)
Encoder MBE:                  0.17 ms  (  0.5%)
Decoder Forward:             10.24 ms  ( 33.4%)
Decoder MBE:                  0.16 ms  (  0.5%)
Output (head + loss):         2.52 ms  (  8.2%)
------------------------------------------------------------
TOTAL:                       30.67 ms



In [6]:
g.grad_info

{'skip_weights': defaultdict(list,
             {'prev_grad_norm': [3.026798367500305e-07,
               3.026798367500305e-07,
               3.026798367500305e-07,
               4.00003045797348e-07,
               4.973262548446655e-07],
              'curr_grad_norm': [0.0,
               0.0,
               9.73232090473175e-08,
               9.73232090473175e-08,
               0.0],
              'cosine_similarity': [0.0, 0.0, 1.0, 1.0, 0.0],
              'loss_name': ['mbe_0', 'mbe_1', 'mbe_2', 'mbe', 'ce'],
              'reset': [False, False, False, False, False]}),
 'transformer.wte.weight': defaultdict(list,
             {'prev_grad_norm': [0.1288611739873886,
               0.13041551411151886,
               0.13465794920921326,
               0.14134655892848969,
               0.17276106774806976],
              'curr_grad_norm': [0.018993979319930077,
               0.018993979319930077,
               0.018993979319930077,
               0.056981924921274185,
  